###PHASE 6 — Cross Encoder Reranking

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q sentence-transformers

In [3]:
import pickle
import numpy as np
from sentence_transformers import CrossEncoder
from tqdm import tqdm

In [4]:
with open(
"/content/drive/MyDrive/Redrob_AI_Hackathon/outputs/top1000_dense.pkl",
"rb"
) as f:

    top1000_dense = pickle.load(f)

print(len(top1000_dense))

1000


In [6]:
model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
query = """
Senior Machine Learning Engineer

AI Engineer

Search Engineer

Recommendation Systems

RAG

Embeddings

Vector Databases

FAISS

Milvus

Pinecone

BM25

PyTorch

Transformers

Semantic Search

Python

Open to work

Strong recruiter response

Short notice period

5 to 9 years experience
"""

In [8]:
pairs = []

for cand in top1000_dense:

    pairs.append(
        [query, cand["text"]]
    )

print(len(pairs))

1000


In [9]:
scores = model.predict(
    pairs,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [10]:
top_indices = np.argsort(scores)[::-1][:300]

In [11]:
top300_cross = []

for idx in top_indices:

    candidate = top1000_dense[idx]

    candidate["cross_score"] = float(scores[idx])

    top300_cross.append(candidate)

In [12]:
with open(
"/content/drive/MyDrive/Redrob_AI_Hackathon/outputs/top300_crossencoder.pkl",
"wb"
) as f:

    pickle.dump(
        top300_cross,
        f
    )

In [13]:
for i in range(10):

    print("="*100)

    print(top300_cross[i]["candidate_id"])

    print()

    print(top300_cross[i]["cross_score"])

    print()

    print(top300_cross[i]["text"][:1500])

CAND_0018499

5.261290550231934

Current role: Senior Machine Learning Engineer.
Primary profession: Senior Machine Learning Engineer.
Current company: Zomato.
Industry: Food Delivery.
Location: Noida, Uttar Pradesh.
Professional experience of 7.2 years.
Professional Summary:
Senior AI engineer with 7.2 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector recall, serving 50M+ queries per month. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I've shipped systems in both early-stage product companies and at larger scale, and I've spent enough time on both that I know which tradeoffs apply where. I have strong opinions about when LLMs are the right hammer and when classical